# Stock Market - Data Cleaning

**Project Name:** Market Time Series Forecasting



**Objective:** This notebook focuses on the initial crucial steps of a data science project: raw stock data fetching, comprehensive data cleaning, validation, and the export of a clean dataset. The goal is to prepare a high-quality, reliable dataset for subsequent analysis or model building, without delving into complex machine learning or forecasting.

---

## Table of Contents

1.  [Project Introduction](#1.-Project-Introduction)
2.  [Install & Import Libraries](#2.-Install-&-Import-Libraries)
3.  [Project Configuration](#3.-Project-Configuration)
4.  [Fetch Data from yfinance](#4.-Fetch-Data-from-yfinance)
5.  [Save Raw Dataset](#5.-Save-Raw-Dataset)
6.  [Initial Dataset Inspection](#6.-Initial-Dataset-Inspection)
    *   [6.1. Display First Rows](#6.1.-Display-First-Rows)
    *   [6.2. Check Dataset Shape](#6.2.-Check-Dataset-Shape)
    *   [6.3. Review Data Information](#6.3.-Review-Data-Information)
    *   [6.4. Generate Descriptive Statistics](#6.4.-Generate-Descriptive-Statistics)
7.  [Missing Value Analysis](#7.-Missing-Value-Analysis)
    *   [7.1. Count Missing Values](#7.1.-Count-Missing-Values)
    *   [7.2. Calculate Missing Value Percentage](#7.2.-Calculate-Missing-Value-Percentage)
    *   [7.3. Handle Missing Values (Forward Fill)](#7.3.-Handle-Missing-Values-(Forward-Fill))
    *   [7.4. Validate Missing Values After Handling](#7.4.-Validate-Missing-Values-After-Handling)
8.  [Duplicate Analysis](#8.-Duplicate-Analysis)
    *   [8.1. Count Duplicate Rows](#8.1.-Count-Duplicate-Rows)
    *   [8.2. Remove Duplicate Rows](#8.2.-Remove-Duplicate-Rows)
    *   [8.3. Verify Duplicates After Removal](#8.3.-Verify-Duplicates-After-Removal)
9.  [Data Cleaning](#9.-Data-Cleaning)
    *   [9.1. Reset Index](#9.1.-Reset-Index)
    *   [9.2. Flatten MultiIndex Columns](#9.2.-Flatten-MultiIndex-Columns)
    *   [9.3. Convert 'Date' to Datetime](#9.3.-Convert-'Date'-to-Datetime)
    *   [9.4. Sort Data by Date](#9.4.-Sort-Data-by-Date)
10. [Datatype Optimization](#10.-Datatype-Optimization)
    *   [10.1. Memory Usage Before Optimization](#10.1.-Memory-Usage-Before-Optimization)
    *   [10.2. Apply Datatype Optimization Logic](#10.2.-Apply-Datatype-Optimization-Logic)
    *   [10.3. Memory Usage After Optimization](#10.3.-Memory-Usage-After-Optimization)
    *   [10.4. Calculate Memory Improvement](#10.4.-Calculate-Memory-Improvement)
11. [Final Validation](#11.-Final-Validation)
    *   [11.1. Final Dataset Shape](#11.1.-Final-Dataset-Shape)
    *   [11.2. Remaining Null Values](#11.2.-Remaining-Null-Values)
    *   [11.3. Remaining Duplicate Count](#11.3.-Remaining-Duplicate-Count)
    *   [11.4. Final Datatype Summary](#11.4.-Final-Datatype-Summary)
    *   [11.5. Final Data Preview](#11.5.-Final-Data-Preview)
12. [Export Clean Dataset](#12.-Export-Clean-Dataset)
13. [Conclusion](#13.-Conclusion)

## 2. Install & Import Libraries

This section ensures that all necessary libraries are installed and imported. We will use `yfinance` for data fetching, `pandas` for data manipulation, and `numpy` for numerical operations.

In [2]:
# Install necessary libraries
!pip install yfinance

# Import standard libraries
import pandas as pd
import numpy as np
import yfinance as yf
import os

print("Libraries installed and imported successfully.")

Libraries installed and imported successfully.


## 3. Project Configuration

Here we define the core parameters for our data fetching and processing. This includes the stock tickers, the time range for historical data, and the paths for saving our raw and cleaned datasets. Centralizing configurations enhances modularity and reproducibility.

In [3]:
# --- Configuration Parameters ---

# List of stock tickers to fetch
STOCK_TICKERS = ['AAPL', 'MSFT', 'SPY']

# Time range for historical data (5 years of daily OHLCV data)
START_DATE = (pd.to_datetime('today') - pd.DateOffset(years=5)).strftime('%Y-%m-%d')
END_DATE = pd.to_datetime('today').strftime('%Y-%m-%d')

# File paths for saving data
RAW_DATA_PATH = 'raw_stock_data.csv'
CLEANED_DATA_PATH = 'cleaned_stock_data.csv'

print(f"Project configured with tickers: {STOCK_TICKERS}, from {START_DATE} to {END_DATE}")

Project configured with tickers: ['AAPL', 'MSFT', 'SPY'], from 2021-07-02 to 2026-07-02


## 4. Fetch Data from yfinance

We utilize the `yfinance` library to download historical OHLCV (Open, High, Low, Close, Volume) data for the specified stock tickers and time range. The data is fetched and stored in a pandas DataFrame, which will have a MultiIndex column structure, common with `yfinance` output for multiple tickers.

In [4]:
# Fetching data for multiple tickers
print(f"Fetching data for {STOCK_TICKERS} from {START_DATE} to {END_DATE}...")
stock_data = yf.download(STOCK_TICKERS, start=START_DATE, end=END_DATE)

# Display the shape and first few rows of the raw data
print(f"Raw data shape: {stock_data.shape}")
display(stock_data.head())

print("Data fetching complete.")

Fetching data for ['AAPL', 'MSFT', 'SPY'] from 2021-07-02 to 2026-07-02...


/tmp/ipykernel_2709/1064002584.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(STOCK_TICKERS, start=START_DATE, end=END_DATE)
[*********************100%***********************]  3 of 3 completed

Raw data shape: (1254, 15)


Price            Close                                High              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-07-02  136.426315  266.459778  405.368103  136.465299  266.795678   
2021-07-06  138.434265  266.469391  404.629700  139.535725  268.110464   
2021-07-07  140.919876  268.647949  406.059692  141.231789  269.377328   
2021-07-08  139.623459  266.239105  402.751068  140.422748  267.496306   
2021-07-09  141.446243  266.738129  407.050385  141.972603  266.843681   

Price                          Low                                Open  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-07-02  405.723267  134.272107  261.517346  402.377273  134.418313   
2021-07-06  405.639117  136.533502  263.244795  401.900594  136.533502   
2021-07-07  406.340099  139.058097  265.979993  403.302549  139.915868   
2021-07-08  403.508117  137.118339  263.791861  399.573301  138.005367   
2021-07-09  407.349473  139.048347  264.223728  402.554813  139.145828   

Price                                  Volume                      
Ticker            MSFT         SPY       AAPL      MSFT       SPY  
Date                                                               
2021-07-02  261.824456  403.452121   78852600  26458000  57697700  
2021-07-06  266.824474  405.424142  108181800  31565600  68710400  
2021-07-07  268.139311  405.311999  104911600  23260000  63549500  
2021-07-08  265.740044  400.750945  105575500  24618600  97595200  
2021-07-09  264.607601  404.255848   99890800  23916700  76238600

Data fetching complete.


## 5. Save Raw Dataset

It is crucial to save the raw, unadulterated data immediately after fetching. This step ensures data reproducibility and serves as a reliable checkpoint, allowing us to revert to the original state if any issues arise during cleaning or transformation. We save the data to `raw_stock_data.csv`.

In [5]:
# Create 'data' directory if it doesn't exist (optional, for better organization)
# os.makedirs('data', exist_ok=True)

# Save the raw data to CSV
stock_data.to_csv(RAW_DATA_PATH)
print(f"Raw data saved to {RAW_DATA_PATH}")

Raw data saved to raw_stock_data.csv


## 6. Initial Dataset Inspection

Before any cleaning, a thorough inspection of the raw dataset is vital. This section provides an overview of the data's structure, types, descriptive statistics, and initial checks for missing values and duplicates. This helps us understand the data characteristics and identify potential issues early on.

In [6]:
print("--- Initial Dataset Inspection ---")

# Display the first few rows of the DataFrame
print("\nHead of the DataFrame:")
display(stock_data.head())

# Display concise summary of the DataFrame, including data types and non-null values
print("\nInfo of the DataFrame:")
stock_data.info()

# Generate descriptive statistics
print("\nDescriptive Statistics:")
display(stock_data.describe())

# Check for missing values
print("\nMissing values per column:")
display(stock_data.isnull().sum())

# Check for duplicate rows
print("\nNumber of duplicate rows:")
display(stock_data.duplicated().sum())

--- Initial Dataset Inspection ---

Head of the DataFrame:


Price            Close                                High              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-07-02  136.426315  266.459778  405.368103  136.465299  266.795678   
2021-07-06  138.434265  266.469391  404.629700  139.535725  268.110464   
2021-07-07  140.919876  268.647949  406.059692  141.231789  269.377328   
2021-07-08  139.623459  266.239105  402.751068  140.422748  267.496306   
2021-07-09  141.446243  266.738129  407.050385  141.972603  266.843681   

Price                          Low                                Open  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-07-02  405.723267  134.272107  261.517346  402.377273  134.418313   
2021-07-06  405.639117  136.533502  263.244795  401.900594  136.533502   
2021-07-07  406.340099  139.058097  265.979993  403.302549  139.915868   
2021-07-08  403.508117  137.118339  263.791861  399.573301  138.005367   
2021-07-09  407.349473  139.048347  264.223728  402.554813  139.145828   

Price                                  Volume                      
Ticker            MSFT         SPY       AAPL      MSFT       SPY  
Date                                                               
2021-07-02  261.824456  403.452121   78852600  26458000  57697700  
2021-07-06  266.824474  405.424142  108181800  31565600  68710400  
2021-07-07  268.139311  405.311999  104911600  23260000  63549500  
2021-07-08  265.740044  400.750945  105575500  24618600  97595200  
2021-07-09  264.607601  404.255848   99890800  23916700  76238600


Info of the DataFrame:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-07-02 to 2026-07-01
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AAPL)   1254 non-null   float64
 1   (Close, MSFT)   1254 non-null   float64
 2   (Close, SPY)    1254 non-null   float64
 3   (High, AAPL)    1254 non-null   float64
 4   (High, MSFT)    1254 non-null   float64
 5   (High, SPY)     1254 non-null   float64
 6   (Low, AAPL)     1254 non-null   float64
 7   (Low, MSFT)     1254 non-null   float64
 8   (Low, SPY)      1254 non-null   float64
 9   (Open, AAPL)    1254 non-null   float64
 10  (Open, MSFT)    1254 non-null   float64
 11  (Open, SPY)     1254 non-null   float64
 12  (Volume, AAPL)  1254 non-null   int64  
 13  (Volume, MSFT)  1254 non-null   int64  
 14  (Volume, SPY)   1254 non-null   int64  
dtypes: float64(12), int64(3)
memory usage: 156.8 KB

Descriptive Statistics:


Price         Close                                   High               \
Ticker         AAPL         MSFT          SPY         AAPL         MSFT   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     193.846162   358.804743   499.761325   195.787048   362.167865   
std       44.823570    83.650674   111.877840    45.213720    83.995882   
min      122.933556   207.734009   339.378540   125.637661   213.706668   
25%      156.551342   282.929016   407.008339   159.102415   284.758098   
50%      182.623787   367.018997   459.318863   184.382766   369.113317   
75%      226.556625   418.293037   586.484940   228.677560   421.501025   
max      315.200012   538.658508   757.618225   317.399994   551.048474   

Price                        Low                                   Open  \
Ticker          SPY         AAPL         MSFT          SPY         AAPL   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     502.418481   191.753195   355.203522   496.651142   193.663197   
std      111.998902    44.478932    83.322518   111.628702    44.846144   
min      342.481461   122.097738   206.938912   331.335684   123.907026   
25%      409.356813   154.411070   279.452476   403.311825   156.818692   
50%      460.930257   180.765384   363.056012   456.421242   182.315155   
75%      589.118044   224.388723   414.080896   583.511208   226.723156   
max      758.446109   309.649994   537.366702   754.805464   314.179993   

Price                                   Volume                              
Ticker         MSFT          SPY          AAPL          MSFT           SPY  
count   1254.000000  1254.000000  1.254000e+03  1.254000e+03  1.254000e+03  
mean     358.818798   499.647526  6.531265e+07  2.655542e+07  7.570563e+07  
std       83.776886   111.880538  2.874232e+07  1.205600e+07  2.910558e+07  
min      210.933620   332.382684  1.791060e+07  5.855900e+06  2.604870e+07  
25%      282.535165   406.999205  4.578078e+07  1.906325e+07  5.663625e+07  
50%      366.305704   458.192778  5.728620e+07  2.364530e+07  7.120585e+07  
75%      418.234862   586.885130  7.740915e+07  3.066400e+07  8.941912e+07  
max      550.830186   756.201867  3.186799e+08  1.862016e+08  2.566114e+08


Missing values per column:


Price   Ticker
Close   AAPL      0
        MSFT      0
        SPY       0
High    AAPL      0
        MSFT      0
        SPY       0
Low     AAPL      0
        MSFT      0
        SPY       0
Open    AAPL      0
        MSFT      0
        SPY       0
Volume  AAPL      0
        MSFT      0
        SPY       0
dtype: int64


Number of duplicate rows:


np.int64(0)

### 6.1. Display First Rows

Examining the head of the DataFrame provides a quick glimpse into the data structure and initial values.

In [7]:
# Display the first few rows of the DataFrame
print("Head of the DataFrame:")
display(stock_data.head())

Head of the DataFrame:


Price            Close                                High              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-07-02  136.426315  266.459778  405.368103  136.465299  266.795678   
2021-07-06  138.434265  266.469391  404.629700  139.535725  268.110464   
2021-07-07  140.919876  268.647949  406.059692  141.231789  269.377328   
2021-07-08  139.623459  266.239105  402.751068  140.422748  267.496306   
2021-07-09  141.446243  266.738129  407.050385  141.972603  266.843681   

Price                          Low                                Open  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-07-02  405.723267  134.272107  261.517346  402.377273  134.418313   
2021-07-06  405.639117  136.533502  263.244795  401.900594  136.533502   
2021-07-07  406.340099  139.058097  265.979993  403.302549  139.915868   
2021-07-08  403.508117  137.118339  263.791861  399.573301  138.005367   
2021-07-09  407.349473  139.048347  264.223728  402.554813  139.145828   

Price                                  Volume                      
Ticker            MSFT         SPY       AAPL      MSFT       SPY  
Date                                                               
2021-07-02  261.824456  403.452121   78852600  26458000  57697700  
2021-07-06  266.824474  405.424142  108181800  31565600  68710400  
2021-07-07  268.139311  405.311999  104911600  23260000  63549500  
2021-07-08  265.740044  400.750945  105575500  24618600  97595200  
2021-07-09  264.607601  404.255848   99890800  23916700  76238600

### 6.2. Check Dataset Shape

Understanding the number of rows and columns helps to gauge the size of the dataset.

In [8]:
# Display the shape of the raw data
print(f"Raw data shape: {stock_data.shape} (rows, columns)")

Raw data shape: (1254, 15) (rows, columns)


### 6.3. Review Data Information

The `info()` method provides a concise summary of the DataFrame, including data types and non-null values for each column.

In [9]:
# Display concise summary of the DataFrame
print("Info of the DataFrame:")
stock_data.info()

Info of the DataFrame:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-07-02 to 2026-07-01
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AAPL)   1254 non-null   float64
 1   (Close, MSFT)   1254 non-null   float64
 2   (Close, SPY)    1254 non-null   float64
 3   (High, AAPL)    1254 non-null   float64
 4   (High, MSFT)    1254 non-null   float64
 5   (High, SPY)     1254 non-null   float64
 6   (Low, AAPL)     1254 non-null   float64
 7   (Low, MSFT)     1254 non-null   float64
 8   (Low, SPY)      1254 non-null   float64
 9   (Open, AAPL)    1254 non-null   float64
 10  (Open, MSFT)    1254 non-null   float64
 11  (Open, SPY)     1254 non-null   float64
 12  (Volume, AAPL)  1254 non-null   int64  
 13  (Volume, MSFT)  1254 non-null   int64  
 14  (Volume, SPY)   1254 non-null   int64  
dtypes: float64(12), int64(3)
memory usage: 156.8 KB


### 6.4. Generate Descriptive Statistics

Descriptive statistics give insights into the central tendency, dispersion, and shape of the dataset's distribution.

In [10]:
# Generate descriptive statistics
print("Descriptive Statistics:")
display(stock_data.describe())

Descriptive Statistics:


Price         Close                                   High               \
Ticker         AAPL         MSFT          SPY         AAPL         MSFT   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     193.846162   358.804743   499.761325   195.787048   362.167865   
std       44.823570    83.650674   111.877840    45.213720    83.995882   
min      122.933556   207.734009   339.378540   125.637661   213.706668   
25%      156.551342   282.929016   407.008339   159.102415   284.758098   
50%      182.623787   367.018997   459.318863   184.382766   369.113317   
75%      226.556625   418.293037   586.484940   228.677560   421.501025   
max      315.200012   538.658508   757.618225   317.399994   551.048474   

Price                        Low                                   Open  \
Ticker          SPY         AAPL         MSFT          SPY         AAPL   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     502.418481   191.753195   355.203522   496.651142   193.663197   
std      111.998902    44.478932    83.322518   111.628702    44.846144   
min      342.481461   122.097738   206.938912   331.335684   123.907026   
25%      409.356813   154.411070   279.452476   403.311825   156.818692   
50%      460.930257   180.765384   363.056012   456.421242   182.315155   
75%      589.118044   224.388723   414.080896   583.511208   226.723156   
max      758.446109   309.649994   537.366702   754.805464   314.179993   

Price                                   Volume                              
Ticker         MSFT          SPY          AAPL          MSFT           SPY  
count   1254.000000  1254.000000  1.254000e+03  1.254000e+03  1.254000e+03  
mean     358.818798   499.647526  6.531265e+07  2.655542e+07  7.570563e+07  
std       83.776886   111.880538  2.874232e+07  1.205600e+07  2.910558e+07  
min      210.933620   332.382684  1.791060e+07  5.855900e+06  2.604870e+07  
25%      282.535165   406.999205  4.578078e+07  1.906325e+07  5.663625e+07  
50%      366.305704   458.192778  5.728620e+07  2.364530e+07  7.120585e+07  
75%      418.234862   586.885130  7.740915e+07  3.066400e+07  8.941912e+07  
max      550.830186   756.201867  3.186799e+08  1.862016e+08  2.566114e+08

---

## 7. Missing Value Analysis

Missing values are common in time series data, especially for financial instruments due to holidays or market closures. We will analyze the extent of missing data and formulate a strategy for handling it. For stock data, forward-filling (ffill) is often appropriate, as stock prices tend to retain their last known value.

### 7.1. Count Missing Values

We start by counting the total number of missing values per column to understand the extent of missingness.

In [11]:
# Make a copy of the original DataFrame for cleaning operations
df_cleaned = stock_data.copy()

# Check for missing values
print("Missing values per column before handling:")
display(df_cleaned.isnull().sum().sort_values(ascending=False))

Missing values per column before handling:


Price   Ticker
Close   AAPL      0
        MSFT      0
        SPY       0
High    AAPL      0
        MSFT      0
        SPY       0
Low     AAPL      0
        MSFT      0
        SPY       0
Open    AAPL      0
        MSFT      0
        SPY       0
Volume  AAPL      0
        MSFT      0
        SPY       0
dtype: int64

### 7.2. Calculate Missing Value Percentage

Calculating percentages provides a relative measure of missingness, which is often more informative than raw counts.

In [12]:
# Calculate missing value percentage
missing_percentage = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100
print("Missing value percentage per column:")
display(missing_percentage.sort_values(ascending=False))

Missing value percentage per column:


Price   Ticker
Close   AAPL      0.0
        MSFT      0.0
        SPY       0.0
High    AAPL      0.0
        MSFT      0.0
        SPY       0.0
Low     AAPL      0.0
        MSFT      0.0
        SPY       0.0
Open    AAPL      0.0
        MSFT      0.0
        SPY       0.0
Volume  AAPL      0.0
        MSFT      0.0
        SPY       0.0
dtype: float64

### 7.3. Handle Missing Values (Forward Fill)

For time series financial data, forward-filling (`ffill`) is a common and appropriate strategy. It propagates the last valid observation forward to next valid observation, which makes sense for stock prices on non-trading days.

In [13]:
print("Applying forward fill to missing values...")

# Get unique tickers from the MultiIndex columns
tickers = df_cleaned.columns.get_level_values(1).unique()

for ticker in tickers:
    # Select all columns for the current ticker and apply ffill
    # Using pd.IndexSlice to correctly select across the MultiIndex
    df_cleaned.loc[:, pd.IndexSlice[:, ticker]] = df_cleaned.loc[:, pd.IndexSlice[:, ticker]].ffill()

print("Forward fill application complete.")

Applying forward fill to missing values...
Forward fill application complete.


### 7.4. Validate Missing Values After Handling

After applying the forward fill, we re-check for any remaining missing values. If there are any, they are likely at the very beginning of the time series for certain stocks, and we will drop those rows to ensure a complete dataset.

In [14]:
print("Missing values after forward fill:")
display(df_cleaned.isnull().sum().sort_values(ascending=False))

# If there are any remaining NaNs (e.g., at the very beginning of the series for some stocks),
# dropping these rows is a clean way to ensure no NaNs remain for analysis.
initial_na_count = df_cleaned.isnull().sum().sum()
if initial_na_count > 0:
    print(f"\n{initial_na_count} remaining NaNs found. Dropping rows with any remaining NaNs.")
    df_cleaned.dropna(inplace=True)
    print(f"DataFrame shape after dropping remaining NaNs: {df_cleaned.shape}")
    print("Missing values after dropping rows (should be all zeros):")
    display(df_cleaned.isnull().sum().sort_values(ascending=False))
else:
    print("No missing values remain after forward fill.")

Missing values after forward fill:


Price   Ticker
Close   AAPL      0
        MSFT      0
        SPY       0
High    AAPL      0
        MSFT      0
        SPY       0
Low     AAPL      0
        MSFT      0
        SPY       0
Open    AAPL      0
        MSFT      0
        SPY       0
Volume  AAPL      0
        MSFT      0
        SPY       0
dtype: int64

No missing values remain after forward fill.


---

In [15]:
print("--- Missing Value Analysis and Handling ---")

# Make a copy of the original DataFrame for cleaning operations
df_cleaned = stock_data.copy()

print("Missing values before handling:")
display(df_cleaned.isnull().sum().sort_values(ascending=False).head(10))

# Correctly apply forward fill to handle missing values
# This correctly handles the MultiIndex columns and assigns back to the DataFrame.
# We iterate through each ticker and ffill its data.

print("Applying forward fill to missing values...")

# Get unique tickers from the MultiIndex columns
tickers = df_cleaned.columns.get_level_values(1).unique()

for ticker in tickers:
    # Select all columns for the current ticker and apply ffill
    # Using pd.IndexSlice to correctly select across the MultiIndex
    df_cleaned.loc[:, pd.IndexSlice[:, ticker]] = df_cleaned.loc[:, pd.IndexSlice[:, ticker]].ffill()

print("Missing values after forward fill:")
display(df_cleaned.isnull().sum().sort_values(ascending=False).head(10))

# After ffill, some rows might still have NaN if the very first values were missing.
# For financial data, it's common to drop rows where critical columns (like 'Open' or 'Close')
# for all tickers are still NaN after ffill, especially if the whole day is missing.
# However, given we are ffilling, these should be minimal or non-existent unless
# the very first observation for a stock is missing.

# If there are any remaining NaNs (e.g., at the very beginning of the series for some stocks),
# we can decide to drop them or fill them with 0/mean. For this context, dropping initial NaNs is safer.
initial_na_count = df_cleaned.isnull().sum().sum()
if initial_na_count > 0:
    print(f"\n{initial_na_count} remaining NaNs found. Dropping rows with any remaining NaNs.")
    df_cleaned.dropna(inplace=True)
    print(f"DataFrame shape after dropping remaining NaNs: {df_cleaned.shape}")
    print("Missing values after dropping rows:")
    display(df_cleaned.isnull().sum().sort_values(ascending=False).head(10))

print("Missing value handling complete.")

--- Missing Value Analysis and Handling ---
Missing values before handling:


Price  Ticker
Close  AAPL      0
       MSFT      0
       SPY       0
High   AAPL      0
       MSFT      0
       SPY       0
Low    AAPL      0
       MSFT      0
       SPY       0
Open   AAPL      0
dtype: int64

Applying forward fill to missing values...
Missing values after forward fill:


Price  Ticker
Close  AAPL      0
       MSFT      0
       SPY       0
High   AAPL      0
       MSFT      0
       SPY       0
Low    AAPL      0
       MSFT      0
       SPY       0
Open   AAPL      0
dtype: int64

Missing value handling complete.


## 8. Duplicate Analysis

Duplicate rows can skew analyses and models. We will check for any exact duplicate rows in our dataset. For time series data, duplicates are often indicative of data ingestion issues rather than actual repeated observations. We expect no duplicates after data fetching from `yfinance` but validate this assumption.

### 8.1. Count Duplicate Rows

Identifying duplicate rows is the first step in ensuring data integrity. Duplicates can lead to biased analyses and incorrect model training.

In [16]:
# Check for duplicate rows
duplicate_rows = df_cleaned.duplicated()
num_duplicates = duplicate_rows.sum()

print(f"Number of duplicate rows found: {num_duplicates}")

Number of duplicate rows found: 0


### 8.2. Remove Duplicate Rows

If any duplicates are found, they are removed to ensure each record is unique. For time series data, duplicates are often an anomaly rather than a valid observation.

In [17]:
if num_duplicates > 0:
    print("Dropping duplicate rows...")
    df_cleaned.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows to remove.")

No duplicate rows to remove.


### 8.3. Verify Duplicates After Removal

After attempting to remove duplicates, we verify that no duplicates remain, confirming the success of the operation.

In [18]:
# Verify for remaining duplicate rows
num_duplicates_after = df_cleaned.duplicated().sum()
print(f"Number of duplicate rows after removal: {num_duplicates_after}")

if num_duplicates_after == 0:
    print("Duplicate analysis and handling complete. No duplicates found.")
else:
    print("Warning: Duplicates still exist after removal attempt.")

Number of duplicate rows after removal: 0
Duplicate analysis and handling complete. No duplicates found.


---

In [19]:
print("--- Duplicate Analysis and Handling ---")

# Check for duplicate rows
duplicate_rows = df_cleaned.duplicated()
num_duplicates = duplicate_rows.sum()

print(f"Number of duplicate rows found: {num_duplicates}")

if num_duplicates > 0:
    print("Dropping duplicate rows...")
    df_cleaned.drop_duplicates(inplace=True)
    print(f"DataFrame shape after dropping duplicates: {df_cleaned.shape}")
else:
    print("No duplicate rows found.")

print("Duplicate handling complete.")

--- Duplicate Analysis and Handling ---
Number of duplicate rows found: 0
No duplicate rows found.
Duplicate handling complete.


## 9. Data Cleaning

This section consolidates all specific cleaning operations beyond just missing values and duplicates. For `yfinance` data, the primary cleaning involves ensuring the MultiIndex columns are handled correctly. We will flatten the MultiIndex columns to make the DataFrame more user-friendly for subsequent analysis. The 'Date' index will also be converted to a column.

### 9.1. Reset Index

Initially, the 'Date' column is part of the DataFrame's index. Resetting the index converts 'Date' into a regular column, which is often more convenient for manipulation.

In [20]:
# Reset index to make 'Date' a regular column
print("Resetting DataFrame index...")
df_cleaned = df_cleaned.reset_index()
print("Index reset. 'Date' is now a column.")
display(df_cleaned.head())

Resetting DataFrame index...
Index reset. 'Date' is now a column.


Price        Date       Close                                High              \
Ticker                   AAPL        MSFT         SPY        AAPL        MSFT   
0      2021-07-02  136.426315  266.459778  405.368103  136.465299  266.795678   
1      2021-07-06  138.434265  266.469391  404.629700  139.535725  268.110464   
2      2021-07-07  140.919876  268.647949  406.059692  141.231789  269.377328   
3      2021-07-08  139.623459  266.239105  402.751068  140.422748  267.496306   
4      2021-07-09  141.446243  266.738129  407.050385  141.972603  266.843681   

Price                      Low                                Open  \
Ticker         SPY        AAPL        MSFT         SPY        AAPL   
0       405.723267  134.272107  261.517346  402.377273  134.418313   
1       405.639117  136.533502  263.244795  401.900594  136.533502   
2       406.340099  139.058097  265.979993  403.302549  139.915868   
3       403.508117  137.118339  263.791861  399.573301  138.005367   
4       407.349473  139.048347  264.223728  402.554813  139.145828   

Price                              Volume                      
Ticker        MSFT         SPY       AAPL      MSFT       SPY  
0       261.824456  403.452121   78852600  26458000  57697700  
1       266.824474  405.424142  108181800  31565600  68710400  
2       268.139311  405.311999  104911600  23260000  63549500  
3       265.740044  400.750945  105575500  24618600  97595200  
4       264.607601  404.255848   99890800  23916700  76238600

### 9.2. Flatten MultiIndex Columns

The `yfinance` library often returns data with MultiIndex columns. To simplify access and improve readability, we flatten these into a single level, creating new column names like 'Open_AAPL', 'High_MSFT', etc.

In [21]:
# Flatten MultiIndex columns
# Create new column names by combining level 0 (e.g., 'Open', 'High') and level 1 (e.g., 'AAPL', 'MSFT')
print("Flattening MultiIndex columns...")
new_columns = []
for col_level_0, col_level_1 in df_cleaned.columns:
    if col_level_0 == 'Date': # Handle the 'Date' column separately
        new_columns.append('Date')
    else:
        new_columns.append(f"{col_level_0}_{col_level_1}")

df_cleaned.columns = new_columns
print("MultiIndex columns flattened.")
display(df_cleaned.head())

Flattening MultiIndex columns...
MultiIndex columns flattened.


,Date,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,Open_MSFT,Open_SPY,Volume_AAPL,Volume_MSFT,Volume_SPY
0,2021-07-02,136.426315,266.459778,405.368103,136.465299,266.795678,405.723267,134.272107,261.517346,402.377273,134.418313,261.824456,403.452121,78852600,26458000,57697700
1,2021-07-06,138.434265,266.469391,404.629700,139.535725,268.110464,405.639117,136.533502,263.244795,401.900594,136.533502,266.824474,405.424142,108181800,31565600,68710400
2,2021-07-07,140.919876,268.647949,406.059692,141.231789,269.377328,406.340099,139.058097,265.979993,403.302549,139.915868,268.139311,405.311999,104911600,23260000,63549500
3,2021-07-08,139.623459,266.239105,402.751068,140.422748,267.496306,403.508117,137.118339,263.791861,399.573301,138.005367,265.740044,400.750945,105575500,24618600,97595200
4,2021-07-09,141.446243,266.738129,407.050385,141.972603,266.843681,407.349473,139.048347,264.223728,402.554813,139.145828,264.607601,404.255848,99890800,23916700,76238600


### 9.3. Convert 'Date' to Datetime

Ensuring the 'Date' column is of `datetime` type is crucial for time series analysis and proper sorting.

In [22]:
# Convert 'Date' column to datetime type
print("Converting 'Date' column to datetime type...")
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'])
print("'Date' column converted.")
display(df_cleaned.info())

Converting 'Date' column to datetime type...
'Date' column converted.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         1254 non-null   datetime64[ns]
 1   Close_AAPL   1254 non-null   float64       
 2   Close_MSFT   1254 non-null   float64       
 3   Close_SPY    1254 non-null   float64       
 4   High_AAPL    1254 non-null   float64       
 5   High_MSFT    1254 non-null   float64       
 6   High_SPY     1254 non-null   float64       
 7   Low_AAPL     1254 non-null   float64       
 8   Low_MSFT     1254 non-null   float64       
 9   Low_SPY      1254 non-null   float64       
 10  Open_AAPL    1254 non-null   float64       
 11  Open_MSFT    1254 non-null   float64       
 12  Open_SPY     1254 non-null   float64       
 13  Volume_AAPL  1254 non-null   int64         
 14  Volume_MSFT  1254 non-null   int64

None

### 9.4. Sort Data by Date

For any time series analysis, it is imperative that the data is sorted chronologically by date. This ensures proper sequencing for subsequent operations.

In [23]:
# Sort by Date for time series consistency
print("Sorting DataFrame by 'Date'...")
df_cleaned.sort_values(by='Date', inplace=True)
print("DataFrame sorted by date.")
display(df_cleaned.head())

Sorting DataFrame by 'Date'...
DataFrame sorted by date.


,Date,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,Open_MSFT,Open_SPY,Volume_AAPL,Volume_MSFT,Volume_SPY
0,2021-07-02,136.426315,266.459778,405.368103,136.465299,266.795678,405.723267,134.272107,261.517346,402.377273,134.418313,261.824456,403.452121,78852600,26458000,57697700
1,2021-07-06,138.434265,266.469391,404.629700,139.535725,268.110464,405.639117,136.533502,263.244795,401.900594,136.533502,266.824474,405.424142,108181800,31565600,68710400
2,2021-07-07,140.919876,268.647949,406.059692,141.231789,269.377328,406.340099,139.058097,265.979993,403.302549,139.915868,268.139311,405.311999,104911600,23260000,63549500
3,2021-07-08,139.623459,266.239105,402.751068,140.422748,267.496306,403.508117,137.118339,263.791861,399.573301,138.005367,265.740044,400.750945,105575500,24618600,97595200
4,2021-07-09,141.446243,266.738129,407.050385,141.972603,266.843681,407.349473,139.048347,264.223728,402.554813,139.145828,264.607601,404.255848,99890800,23916700,76238600


---

In [25]:
print("--- Data Cleaning and Restructuring ---")

# The following steps (reset_index and flatten MultiIndex columns)
# have already been performed in previous dedicated cells (9.1 and 9.2).
# Running them again on an already processed DataFrame leads to errors
# or incorrect data structure. Therefore, they are removed from this summary cell.

# Convert 'Date' column to datetime type (re-applying is harmless)
print("Converting 'Date' column to datetime type...")
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'])
print("'Date' column converted.")

# Sort by Date for time series consistency (re-applying is harmless)
print("Sorting DataFrame by 'Date'...")
df_cleaned.sort_values(by='Date', inplace=True)
print("DataFrame sorted by date.")

print("Cleaned DataFrame head:")
display(df_cleaned.head())
print("Cleaned DataFrame info:")
df_cleaned.info()

print("Data cleaning complete.")

--- Data Cleaning and Restructuring ---
Converting 'Date' column to datetime type...
'Date' column converted.
Sorting DataFrame by 'Date'...
DataFrame sorted by date.
Cleaned DataFrame head:


,index,Date,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,Open_MSFT,Open_SPY,Volume_AAPL,Volume_MSFT,Volume_SPY
0,0,2021-07-02,136.426315,266.459778,405.368103,136.465299,266.795678,405.723267,134.272107,261.517346,402.377273,134.418313,261.824456,403.452121,78852600,26458000,57697700
1,1,2021-07-06,138.434265,266.469391,404.629700,139.535725,268.110464,405.639117,136.533502,263.244795,401.900594,136.533502,266.824474,405.424142,108181800,31565600,68710400
2,2,2021-07-07,140.919876,268.647949,406.059692,141.231789,269.377328,406.340099,139.058097,265.979993,403.302549,139.915868,268.139311,405.311999,104911600,23260000,63549500
3,3,2021-07-08,139.623459,266.239105,402.751068,140.422748,267.496306,403.508117,137.118339,263.791861,399.573301,138.005367,265.740044,400.750945,105575500,24618600,97595200
4,4,2021-07-09,141.446243,266.738129,407.050385,141.972603,266.843681,407.349473,139.048347,264.223728,402.554813,139.145828,264.607601,404.255848,99890800,23916700,76238600


Cleaned DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   index        1254 non-null   int64         
 1   Date         1254 non-null   datetime64[ns]
 2   Close_AAPL   1254 non-null   float64       
 3   Close_MSFT   1254 non-null   float64       
 4   Close_SPY    1254 non-null   float64       
 5   High_AAPL    1254 non-null   float64       
 6   High_MSFT    1254 non-null   float64       
 7   High_SPY     1254 non-null   float64       
 8   Low_AAPL     1254 non-null   float64       
 9   Low_MSFT     1254 non-null   float64       
 10  Low_SPY      1254 non-null   float64       
 11  Open_AAPL    1254 non-null   float64       
 12  Open_MSFT    1254 non-null   float64       
 13  Open_SPY     1254 non-null   float64       
 14  Volume_AAPL  1254 non-null   int64         
 15  Volume_MSFT  1254 non-null   in

## 10. Datatype Optimization

Optimizing data types can significantly reduce memory usage, which is crucial for large datasets. We will convert `float64` columns to `float32` and `int64` columns to `int32` where appropriate, carefully preserving data integrity. We will also compare memory usage before and after optimization to demonstrate its effectiveness.

### 10.1. Memory Usage Before Optimization

We start by calculating the current memory footprint of the DataFrame. This baseline will help us quantify the impact of our optimization efforts.

In [26]:
# Calculate memory usage before optimization
memory_before = df_cleaned.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage before optimization: {memory_before:.2f} MB")

Memory usage before optimization: 0.16 MB


### 10.2. Apply Datatype Optimization Logic

We systematically convert `float64` columns to `float32` and `int64` columns to `int32` where the data range permits. This reduces memory consumption without losing significant precision for financial data.

In [27]:
print("Applying datatype optimization...")
# Iterate through columns to optimize data types
for col in df_cleaned.columns:
    if col == 'Date':
        continue # Skip date column, already optimized to datetime

    if df_cleaned[col].dtype == 'float64':
        df_cleaned[col] = df_cleaned[col].astype('float32')
    elif df_cleaned[col].dtype == 'int64':
        # Check if int32 can safely store the values
        if df_cleaned[col].min() >= np.iinfo(np.int32).min and \
           df_cleaned[col].max() <= np.iinfo(np.int32).max:
            df_cleaned[col] = df_cleaned[col].astype('int32')
        else:
            print(f"Warning: Column '{col}' cannot be safely converted to int32 due to range.")
    # No other types are expected to be optimized in this context (e.g., 'object' for strings)

print("Datatype optimization logic applied.")

Applying datatype optimization...
Datatype optimization logic applied.


### 10.3. Memory Usage After Optimization

After applying the optimization, we recalculate the memory usage to see the effect.

In [28]:
# Calculate memory usage after optimization
memory_after = df_cleaned.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage after optimization: {memory_after:.2f} MB")

Memory usage after optimization: 0.09 MB


### 10.4. Calculate Memory Improvement

Finally, we quantify the percentage reduction in memory usage, demonstrating the efficiency gains from datatype optimization.

In [29]:
print(f"Memory reduced by: {(memory_before - memory_after):.2f} MB ({(1 - (memory_after / memory_before)) * 100:.2f}%) approx.")
print("Datatype optimization complete.")

Memory reduced by: 0.08 MB (47.02%) approx.
Datatype optimization complete.


---

In [30]:
print("--- Datatype Optimization ---")

# Calculate memory usage before optimization
memory_before = df_cleaned.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage before optimization: {memory_before:.2f} MB")

# Iterate through columns to optimize data types
for col in df_cleaned.columns:
    if col == 'Date':
        continue # Skip date column, already optimized to datetime

    if df_cleaned[col].dtype == 'float64':
        df_cleaned[col] = df_cleaned[col].astype('float32')
    elif df_cleaned[col].dtype == 'int64':
        # Check if int32 can safely store the values
        if df_cleaned[col].min() >= np.iinfo(np.int32).min and \
           df_cleaned[col].max() <= np.iinfo(np.int32).max:
            df_cleaned[col] = df_cleaned[col].astype('int32')
        else:
            print(f"Warning: Column '{col}' cannot be safely converted to int32 due to range.")
    # No other types are expected to be optimized in this context (e.g., 'object' for strings)

# Calculate memory usage after optimization
memory_after = df_cleaned.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage after optimization: {memory_after:.2f} MB")
print(f"Memory reduced by: {(memory_before - memory_after):.2f} MB ({(1 - (memory_after / memory_before)) * 100:.2f}%) approx.")

print("Datatype optimization complete. Optimized DataFrame info:")
df_cleaned.info()

--- Datatype Optimization ---
Memory usage before optimization: 0.09 MB
Memory usage after optimization: 0.09 MB
Memory reduced by: 0.00 MB (0.00%) approx.
Datatype optimization complete. Optimized DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   index        1254 non-null   int32         
 1   Date         1254 non-null   datetime64[ns]
 2   Close_AAPL   1254 non-null   float32       
 3   Close_MSFT   1254 non-null   float32       
 4   Close_SPY    1254 non-null   float32       
 5   High_AAPL    1254 non-null   float32       
 6   High_MSFT    1254 non-null   float32       
 7   High_SPY     1254 non-null   float32       
 8   Low_AAPL     1254 non-null   float32       
 9   Low_MSFT     1254 non-null   float32       
 10  Low_SPY      1254 non-null   float32       
 11  Open_AAPL    1254 non-null   float32 

## 11. Final Validation

This final validation step provides a summary of the cleaned dataset, confirming that all cleaning operations have been successful. We check the final shape, ensure no remaining null values, confirm the absence of duplicates, and review the data types. This serves as a quality control checkpoint before exporting the data.

### 11.1. Final Dataset Shape

Confirming the final dimensions of the DataFrame ensures consistency after all cleaning and processing steps.

In [31]:
# Final shape of the DataFrame
print(f"Final DataFrame shape: {df_cleaned.shape} (rows, columns)")

Final DataFrame shape: (1254, 17) (rows, columns)


### 11.2. Remaining Null Values

This check confirms that our missing value handling was successful and no nulls remain in the critical columns.

In [32]:
# Check for remaining null values
print("Remaining null values per column (should be 0):")
display(df_cleaned.isnull().sum())

Remaining null values per column (should be 0):


,0
index,0
Date,0
Close_AAPL,0
Close_MSFT,0
Close_SPY,0
High_AAPL,0
High_MSFT,0
High_SPY,0
Low_AAPL,0
Low_MSFT,0


### 11.3. Remaining Duplicate Count

Verifying the absence of duplicates reassures us of the uniqueness of each record in the dataset.

In [33]:
# Check for remaining duplicate rows
print("Remaining number of duplicate rows (should be 0):")
display(df_cleaned.duplicated().sum())

Remaining number of duplicate rows (should be 0):


np.int64(0)

### 11.4. Final Datatype Summary

A final review of data types confirms that the optimization steps have been correctly applied and the data types are suitable for further analysis.

In [34]:
# Summary of data types
print("Final Datatype Summary:")
df_cleaned.info()

Final Datatype Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   index        1254 non-null   int32         
 1   Date         1254 non-null   datetime64[ns]
 2   Close_AAPL   1254 non-null   float32       
 3   Close_MSFT   1254 non-null   float32       
 4   Close_SPY    1254 non-null   float32       
 5   High_AAPL    1254 non-null   float32       
 6   High_MSFT    1254 non-null   float32       
 7   High_SPY     1254 non-null   float32       
 8   Low_AAPL     1254 non-null   float32       
 9   Low_MSFT     1254 non-null   float32       
 10  Low_SPY      1254 non-null   float32       
 11  Open_AAPL    1254 non-null   float32       
 12  Open_MSFT    1254 non-null   float32       
 13  Open_SPY     1254 non-null   float32       
 14  Volume_AAPL  1254 non-null   int32         
 15  Volume_MSFT  1254 non-null   in

### 11.5. Final Data Preview

A final look at the head of the cleaned DataFrame provides visual confirmation of the data's readiness.

In [35]:
# Display the first few rows of the cleaned DataFrame
print("Final cleaned DataFrame head:")
display(df_cleaned.head())

Final cleaned DataFrame head:


,index,Date,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,Open_MSFT,Open_SPY,Volume_AAPL,Volume_MSFT,Volume_SPY
0,0,2021-07-02,136.426315,266.459778,405.368103,136.465302,266.795685,405.723267,134.272110,261.517334,402.377258,134.418320,261.824463,403.452118,78852600,26458000,57697700
1,1,2021-07-06,138.434265,266.469391,404.629700,139.535721,268.110474,405.639130,136.533508,263.244781,401.900604,136.533508,266.824463,405.424133,108181800,31565600,68710400
2,2,2021-07-07,140.919876,268.647949,406.059692,141.231796,269.377319,406.340088,139.058090,265.979980,403.302551,139.915863,268.139313,405.312012,104911600,23260000,63549500
3,3,2021-07-08,139.623459,266.239105,402.751068,140.422745,267.496307,403.508118,137.118332,263.791870,399.573303,138.005371,265.740051,400.750946,105575500,24618600,97595200
4,4,2021-07-09,141.446243,266.738129,407.050385,141.972595,266.843689,407.349487,139.048340,264.223724,402.554810,139.145828,264.607605,404.255859,99890800,23916700,76238600


---

In [36]:
print("--- Final Validation of Cleaned Dataset ---")

# Final shape of the DataFrame
print(f"\nFinal DataFrame shape: {df_cleaned.shape}")

# Check for remaining null values
print("\nRemaining null values per column (should be 0):")
display(df_cleaned.isnull().sum())

# Check for remaining duplicate rows
print("\nRemaining number of duplicate rows (should be 0):")
display(df_cleaned.duplicated().sum())

# Summary of data types
print("\nFinal Datatype Summary:")
df_cleaned.info()

print("\nFinal validation complete. Dataset is clean and ready for export.")

--- Final Validation of Cleaned Dataset ---

Final DataFrame shape: (1254, 17)

Remaining null values per column (should be 0):


,0
index,0
Date,0
Close_AAPL,0
Close_MSFT,0
Close_SPY,0
High_AAPL,0
High_MSFT,0
High_SPY,0
Low_AAPL,0
Low_MSFT,0



Remaining number of duplicate rows (should be 0):


np.int64(0)


Final Datatype Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   index        1254 non-null   int32         
 1   Date         1254 non-null   datetime64[ns]
 2   Close_AAPL   1254 non-null   float32       
 3   Close_MSFT   1254 non-null   float32       
 4   Close_SPY    1254 non-null   float32       
 5   High_AAPL    1254 non-null   float32       
 6   High_MSFT    1254 non-null   float32       
 7   High_SPY     1254 non-null   float32       
 8   Low_AAPL     1254 non-null   float32       
 9   Low_MSFT     1254 non-null   float32       
 10  Low_SPY      1254 non-null   float32       
 11  Open_AAPL    1254 non-null   float32       
 12  Open_MSFT    1254 non-null   float32       
 13  Open_SPY     1254 non-null   float32       
 14  Volume_AAPL  1254 non-null   int32         
 15  Volume_MSFT  1254 non-null   i

## 12. Export Clean Dataset

With the data thoroughly cleaned and validated, the final step is to export the processed dataset. This clean dataset, stored in `cleaned_stock_data.csv`, can now be used for further analysis, feature engineering, or direct input into machine learning models.

In [37]:
# Export the cleaned DataFrame to CSV
df_cleaned.to_csv(CLEANED_DATA_PATH, index=False)
print(f"Cleaned data successfully exported to {CLEANED_DATA_PATH}")

Cleaned data successfully exported to cleaned_stock_data.csv


## 13. Conclusion

This notebook has successfully demonstrated a complete workflow for fetching raw stock market data from Yahoo Finance, performing comprehensive data cleaning, validating the integrity of the data, and optimizing data types for efficiency. The resulting `cleaned_stock_data.csv` dataset is now prepared for advanced analytical tasks, ensuring a robust foundation for any subsequent financial modeling or time series forecasting efforts.

---